In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import numpy as np


# =========================================
# Least Squares
# =========================================

def least_squares(z, y):

    # Make sure every z_i is a 1D array
    z = [np.asarray(zi).flatten() for zi in z]
    y = np.asarray(y).flatten()

    # Step 1: s = sum(z_i * y_i)
    s = np.zeros(len(z[0]))

    for zi, yi in zip(z, y):
        s += zi * yi

    # Step 2: M = sum(z_i * z_i^T)
    M = np.zeros((len(z[0]), len(z[0])))

    for zi in z:
        M += np.outer(zi, zi)

    # Step 3: Solve Mw = s
    try:
        w = np.linalg.solve(M, s)
    except np.linalg.LinAlgError:
        print("M is singular, using pseudo-inverse")
        w = np.linalg.pinv(M) @ s

    # Step 4: Calculate residual error
    R = 0

    for zi, yi in zip(z, y):
        R += (w @ zi - yi) ** 2

    return w, R


# =========================================
# Polynomial Basis
# =========================================

def polynomial_basis(X, p):

    z = []

    for xi in X:

        # Constant term
        zi = [1]

        # Add x, x^2, ..., x^p
        for degree in range(1, p + 1):

            for feature in xi:
                zi.append(feature ** degree)

        z.append(np.array(zi))

    return z


# =========================================
# Calculate Error
# =========================================

def calculate_error(w, z, y):

    R = 0

    for zi, yi in zip(z, y):

        zi = np.asarray(zi).flatten()

        R += (w @ zi - yi) ** 2

    return R


# =========================================
# Read Data
# =========================================

traindata = np.loadtxt("traindata.txt")

# First 8 columns = input
X = traindata[:, 0:8]

# Last column = output
Y = traindata[:, 8]

print("Original data:")
print("X shape =", X.shape)
print("Y shape =", Y.shape)


# =========================================
# 70/30 Train-Test Split
# =========================================

n = len(X)

split = int(0.7 * n)

X_train = X[:split]
Y_train = Y[:split]

X_test = X[split:]
Y_test = Y[split:]

print("\nTraining data:")
print("X_train =", X_train.shape)
print("Y_train =", Y_train.shape)

print("\nTest data:")
print("X_test =", X_test.shape)
print("Y_test =", Y_test.shape)


# =========================================
# Polynomial Selection
# =========================================

results = []

print("\nPolynomial Selection")
print("====================")

for p in range(0, 11):

    # -----------------------------
    # Create polynomial features
    # -----------------------------

    Z_train = polynomial_basis(
        X_train,
        p
    )

    Z_test = polynomial_basis(
        X_test,
        p
    )

    # -----------------------------
    # Fit polynomial using training
    # -----------------------------

    w, R_train = least_squares(
        Z_train,
        Y_train
    )

    # -----------------------------
    # Calculate test error
    # -----------------------------

    R_test = calculate_error(
        w,
        Z_test,
        Y_test
    )

    # -----------------------------
    # Store result
    # -----------------------------

    results.append(
        (p, R_train, R_test)
    )

    print(
        "p =", p,
        "| train error =", R_train,
        "| test error =", R_test
    )


# =========================================
# Find Best Polynomial
# =========================================

best_result = min(
    results,
    key=lambda result: result[2]
)

best_p = best_result[0]
best_train_error = best_result[1]
best_test_error = best_result[2]


print("\n====================")
print("Best polynomial")
print("====================")

print("Best p =", best_p)
print("Train error =", best_train_error)
print("Test error =", best_test_error)


# =========================================
# Re-fit Best Polynomial on ALL Data
# =========================================

Z_all = polynomial_basis(
    X,
    best_p
)

w_final, R_final = least_squares(
    Z_all,
    Y
)


print("\n====================")
print("Final Model")
print("====================")

print("Polynomial order =", best_p)

print("\nWeights:")
print(w_final)

print("\nTraining residual error:")
print(R_final)